# Nehalennia data layer

In [ ]:
# %pip install pandas openpyxl rdflib

from pathlib import Path

import pandas as pd

import collectie_zeeland
import rmo

OUT_DIR = Path("out")
OUT_DIR.mkdir(exist_ok=True)

EXCEL_CELL_LIMIT = 32767
LINK_COLUMNS = ["pid", "image_url", "iiif_manifest", "afbeelding_url"]


def write_excel(path, sheets):
    """One sheet per {name: frame}, with URL columns turned into clickable links."""
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for name, frame in sheets.items():
            # Excel rejects anything longer than 32767 characters in a single cell
            trimmed = frame.map(lambda v: v[:EXCEL_CELL_LIMIT] if isinstance(v, str) else v)
            trimmed.to_excel(writer, sheet_name=name, index=False)

        for sheet in writer.sheets.values():
            sheet.freeze_panes = "A2"
            sheet.auto_filter.ref = sheet.dimensions
            header = [c.value for c in sheet[1]]
            for column in LINK_COLUMNS:
                if column not in header:
                    continue
                index = header.index(column) + 1
                for (cell,) in sheet.iter_rows(min_row=2, min_col=index, max_col=index):
                    if cell.value:
                        cell.hyperlink = cell.value
                        cell.style = "Hyperlink"
    print(f"wrote {path.resolve()}")

## RMO

First run reads ~400 object pages at one request every half second, so allow a few
minutes; after that everything comes from `out/raw/` and `out/pages/`.

In [ ]:
df_rmo, df_rmo_images = rmo.fetch(cache_dir=OUT_DIR)

write_excel(OUT_DIR / "rmo_nehalennia_colijnsplaat.xlsx",
            {"records": df_rmo, "images": df_rmo_images})
df_rmo.head()

## Collectie Zeeland

Streams the dump on every run (no cache), which takes about 5 seconds.

In [ ]:
df_zeeland = collectie_zeeland.fetch(ttl_path=OUT_DIR / "zeeland_nehalennia.ttl")

write_excel(OUT_DIR / "zeeland_nehalennia.xlsx", {"objects": df_zeeland})
df_zeeland[["objectnummer", "titel", "objecttype", "materiaal", "vervaardiger"]]